# AGNIWATCH Full Pipeline (Colab)
This notebook runs the end-to-end AGNIWATCH pipeline: Sentinel-2 burn mapping, FIRMS active fire stats, Sentinel-5P air quality, emissions, and bulletin generation.

In [ ]:
!pip install -q earthengine-api geemap plotly pandas numpy matplotlib pyyaml

In [ ]:
import ee
from datetime import datetime

from core.config import RegionConfig
from core.preprocessing import get_s2_composite, get_cropland_mask
from core.indices import compute_all_indices
from core.classification import classify_burn_severity, apply_masks, compute_area_stats
from core.firms import get_firms_stats, get_multi_year_trend
from core.airquality import get_s5p_stats, get_monthly_series
from core.emissions import calculate_emissions
from core.alerting import generate_bulletin

In [ ]:
GEE_PROJECT = "your-project-id"
REGION = RegionConfig(
    name="Punjab & Haryana, India",
    bounds=[73.5, 29.5, 77.5, 32.5],
    country="IN",
    crop="Rice (Kharif)",
    pre_start="09-01",
    pre_end="09-30",
    post_start="10-15",
    post_end="11-30",
    sub_regions={
        "Amritsar": [74.5, 31.3, 75.4, 32.2],
        "Ludhiana": [75.5, 30.5, 76.3, 31.2],
        "Patiala": [76.0, 29.9, 76.8, 30.8]
    }
)
YEAR = 2023

In [ ]:
SCOPES = [
    "https://www.googleapis.com/auth/earthengine",
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/devstorage.full_control",
]

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="notebook", scopes=SCOPES, force=False)
    ee.Initialize(project=GEE_PROJECT)

roi = ee.Geometry.Rectangle(REGION.bounds)
pre_start = f"{YEAR}-{REGION.pre_start}"
pre_end = f"{YEAR}-{REGION.pre_end}"
post_start = f"{YEAR}-{REGION.post_start}"
post_end = f"{YEAR}-{REGION.post_end}"
print("Initialized", REGION.name, post_start, post_end)

In [ ]:
pre, pre_count = get_s2_composite(roi, pre_start, pre_end, REGION, "Pre")
post, post_count = get_s2_composite(roi, post_start, post_end, REGION, "Post")
if pre is None or post is None:
    raise RuntimeError("No Sentinel-2 imagery found for selected windows")

idx = compute_all_indices(pre, post)
severity = classify_burn_severity(idx["dnbr"])
crop_mask = get_cropland_mask(roi, YEAR)
dnbr_m, sev_m = apply_masks(idx["dnbr"], severity, idx["ndwi_pre"], crop_mask)
area_stats = compute_area_stats(dnbr_m, sev_m, roi, REGION)

print("S2 images:", pre_count, post_count)
print("Burned area any (km2):", round(area_stats["area_any_km2"], 1))
print("Burned area moderate+ (km2):", round(area_stats["area_moderate_km2"], 1))

In [ ]:
emissions = calculate_emissions(area_stats["area_moderate_km2"], REGION)
firms_stats = get_firms_stats(roi, post_start, post_end, REGION.firms_temp_k)
trend_df = get_multi_year_trend(roi, [2020, 2021, 2022, 2023, 2024], REGION.firms_temp_k)
aq_pre = get_s5p_stats(roi, pre_start, pre_end, REGION.scale_s5p)
aq_post = get_s5p_stats(roi, post_start, post_end, REGION.scale_s5p)
monthly = get_monthly_series(
    roi,
    [
        (f"Aug {YEAR}", f"{YEAR}-08-01", f"{YEAR}-08-31"),
        (f"Sep {YEAR}", f"{YEAR}-09-01", f"{YEAR}-09-30"),
        (f"Oct {YEAR}", f"{YEAR}-10-01", f"{YEAR}-10-31"),
        (f"Nov {YEAR}", f"{YEAR}-11-01", f"{YEAR}-11-30"),
        (f"Dec {YEAR}", f"{YEAR}-12-01", f"{YEAR}-12-31"),
    ],
    REGION.scale_s5p,
)

bulletin = generate_bulletin(REGION, area_stats, firms_stats, aq_pre, aq_post, emissions, YEAR)

print("PM2.5 tonnes:", round(emissions.pm25_tonnes, 1))
print("CO2eq Mt:", round(emissions.co2_eq_million_tonnes, 3))
print("Alert level:", bulletin.alert_level)
print("FIRMS area km2:", firms_stats["area_km2"])
display(trend_df)
display(monthly.head())

In [ ]:
import os
import pandas as pd

os.makedirs("agniwatch_outputs", exist_ok=True)
pd.DataFrame([
    ["region", REGION.name],
    ["season", f"{post_start} -> {post_end}"],
    ["area_any_km2", area_stats["area_any_km2"]],
    ["area_moderate_km2", area_stats["area_moderate_km2"]],
    ["firms_area_km2", firms_stats["area_km2"]],
    ["pm25_tonnes", emissions.pm25_tonnes],
    ["co2_eq_tonnes", emissions.co2_eq_tonnes],
    ["health_cost_usd", emissions.health_cost_usd],
], columns=["metric", "value"]).to_csv("agniwatch_outputs/summary.csv", index=False)

with open("agniwatch_outputs/alert_bulletin.json", "w", encoding="utf-8") as f:
    f.write(bulletin.to_json())

print("Wrote outputs to agniwatch_outputs/")